In [1]:
!pip install -q transformers sentencepiece psutil scikit-learn
print("Done!")

Done!


In [5]:
from google.colab import files
uploaded = files.upload()  # upload baseline_model.zip AND results.zip

!unzip -q baseline_model.zip -d /content/content/
!unzip -q results.zip -d /content/content/
print("Done!")
!ls /content/results/

Saving results.zip to results (1).zip
Saving baseline_model.zip to baseline_model (1).zip
Done!
ls: cannot access '/content/results/': No such file or directory


In [6]:
!ls /content/content/results/

all_results.json  test.csv  train.csv


In [9]:
import os, time, json, gc
import numpy as np
import torch
import torch.nn.utils.prune as prune
import psutil
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR  = '/content/content/baseline_model'
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_LABELS = 7
label2id   = {'neutral':0,'joy':1,'anger':2,'surprise':3,'sadness':4,'fear':5,'disgust':6}

with open('/content/content/results/all_results.json') as f:
    ALL = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, keep_accents=True)
test_df   = pd.read_csv('/content/content/results/test.csv')
print("Device:", device)
print("Results loaded:", list(ALL.keys()))

Device: cuda
Results loaded: ['params', 'baseline_f1', 'baseline_size_mb', 'baseline_latency_ms', 'baseline_ram_mb']


In [10]:
class HindiDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts  = df['Sentence'].tolist()
        self.labels = df['label'].tolist()

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], truncation=True,
            max_length=MAX_LENGTH, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

test_loader = DataLoader(HindiDataset(test_df), batch_size=BATCH_SIZE)
print("Test loader ready. Batches:", len(test_loader))

Test loader ready. Batches: 100


In [11]:
def benchmark_pruned(model, name, save_path):
    model.eval()
    dev = next(model.parameters()).device

    preds_all, labs_all = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop('labels').to(dev)
            batch  = {k: v.to(dev) for k, v in batch.items()}
            preds  = model(**batch).logits.argmax(dim=-1).cpu().numpy()
            preds_all.extend(preds)
            labs_all.extend(labels.cpu().numpy())
    f1 = f1_score(labs_all, preds_all, average='macro')

    # Actual sparsity
    zeros = sum((p == 0).sum().item() for p in model.parameters())
    total = sum(p.numel() for p in model.parameters())

    torch.save(model.state_dict(), save_path)
    size_mb = os.path.getsize(save_path) / 1024**2

    sample = tokenizer(
        "यह बहुत अच्छा है।", return_tensors='pt',
        truncation=True, max_length=MAX_LENGTH, padding='max_length'
    ).to(dev)
    for _ in range(10):
        with torch.no_grad(): _ = model(**sample)
    times = []
    for _ in range(100):
        t = time.perf_counter()
        with torch.no_grad(): _ = model(**sample)
        times.append((time.perf_counter() - t) * 1000)

    result = {
        'f1':              round(f1, 4),
        'size_mb':         round(size_mb, 1),
        'latency_ms':      round(float(np.mean(times)), 2),
        'actual_sparsity': round(zeros/total, 4)
    }
    print(f"\n[{name}]", result)
    return result

In [13]:
for ratio in [0.30, 0.50, 0.70]:
    print(f"\n{'='*40}")
    print(f"Pruning at {int(ratio*100)}%")

    # Fresh model every time
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR, num_labels=NUM_LABELS
    ).to(device)

    # Collect all Linear layers
    to_prune = [
        (mod, 'weight')
        for _, mod in model.named_modules()
        if isinstance(mod, torch.nn.Linear)
    ]
    print(f"Linear layers found: {len(to_prune)}")

    # Global L1 unstructured pruning
    prune.global_unstructured(
        to_prune,
        pruning_method=prune.L1Unstructured,
        amount=ratio
    )

    # Make permanent
    for mod, name in to_prune:
        prune.remove(mod, name)

    key       = f'pruned_{int(ratio*100)}pct'
    save_path = f'/content/content/results/model_pruned_{int(ratio*100)}.pt'
    result    = benchmark_pruned(model, f'Pruned {int(ratio*100)}%', save_path)
    ALL[key]  = result

    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\n✅ All pruning levels done!")
print(json.dumps({k: v for k, v in ALL.items() if 'pruned' in k}, indent=2))


Pruning at 30%


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Linear layers found: 9

[Pruned 30%] {'f1': 0.2406, 'size_mb': 127.6, 'latency_ms': 10.46, 'actual_sparsity': 0.0697}

Pruning at 50%


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Linear layers found: 9

[Pruned 50%] {'f1': 0.0429, 'size_mb': 127.6, 'latency_ms': 11.49, 'actual_sparsity': 0.1162}

Pruning at 70%


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

Linear layers found: 9

[Pruned 70%] {'f1': 0.0298, 'size_mb': 127.6, 'latency_ms': 11.89, 'actual_sparsity': 0.1626}

✅ All pruning levels done!
{
  "pruned_30pct": {
    "f1": 0.2406,
    "size_mb": 127.6,
    "latency_ms": 10.46,
    "actual_sparsity": 0.0697
  },
  "pruned_50pct": {
    "f1": 0.0429,
    "size_mb": 127.6,
    "latency_ms": 11.49,
    "actual_sparsity": 0.1162
  },
  "pruned_70pct": {
    "f1": 0.0298,
    "size_mb": 127.6,
    "latency_ms": 11.89,
    "actual_sparsity": 0.1626
  }
}


In [15]:
with open('/content/content/results/all_results.json', 'w') as f:
    json.dump(ALL, f, indent=2)

!zip -r /content/results.zip /content/results
from google.colab import files
files.download('/content/results.zip')
print("✅ Notebook 3 done!")

	zip warning: name not matched: /content/results

zip error: Nothing to do! (try: zip -r /content/results.zip . -i /content/results)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Notebook 3 done!
